# Lab 3 — Holes and high dimensions

**Accompanies Lecture 3 (and the simplicial homology from Lecture 2).**

You will build the machinery that counts holes, and then measure with your own code what happens to geometry when the dimension grows.

| # | Exercise | What you build |
|---|---|---|
| 1 | Boundary matrices and Betti numbers | `boundary_matrix`, `betti` |
| 2 | Molecules | clique complexes of molecular graphs, $\beta_1$ of rings |
| 3 | Volumes | the ball-in-cube fraction, exactly and by Monte Carlo |
| 4 | Distances | how distances behave as $d$ grows |
| 5 | Real data against its shuffled copy | is real data really low-dimensional? |

Every exercise ends with an `assert` cell. If your code is correct, the notebook runs from top to bottom with no error. Exercises 1–4 are meant for the lab session, Exercise 5 can be finished at home.


In [ ]:
import math, itertools
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA

NAVY, TEAL, AMBER, ICE = "#1E2761", "#1C7293", "#E8871E", "#CADCFC"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("numpy", np.__version__, "| networkx", nx.__version__)


## Exercise 1 — Boundary matrices and Betti numbers

A simplicial complex is stored as a dictionary: dimension $\to$ list of simplices, each simplex a tuple of vertices in increasing order.

**`boundary_matrix(cx, k)`**: rows are the $(k-1)$-simplices `cx[k-1]`, columns are the $k$-simplices `cx[k]`, both in the given order. For a simplex $s=(v_0,\dots,v_k)$, deleting the vertex at position $i$ gives a face, and that face gets the sign $(-1)^i$. For $(1,2,3)$:

- delete position 0 $\to$ $(2,3)$ with $+1$,
- delete position 1 $\to$ $(1,3)$ with $-1$,
- delete position 2 $\to$ $(1,2)$ with $+1$,

which is the column $(1,-1,1)^T$ from the lecture.

**`betti(cx)`**: returns the list $[\beta_0,\dots,\beta_{\text{top}}]$ with
$$\beta_k=(\#k\text{-simplices})-\operatorname{rank}\partial_k-\operatorname{rank}\partial_{k+1},$$
where $\partial_0$ and $\partial_{\text{top}+1}$ do not exist and count as rank $0$.


In [ ]:
# GIVEN: four small complexes and a rank helper
V3 = [(1,), (2,), (3,)]
E3 = [(1, 2), (1, 3), (2, 3)]
hollow_triangle = {0: V3, 1: E3}
filled_triangle = {0: V3, 1: E3, 2: [(1, 2, 3)]}

V4 = [(1,), (2,), (3,), (4,)]
E4 = [(1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)]
F4 = [(1, 2, 3), (1, 2, 4), (1, 3, 4), (2, 3, 4)]
hollow_tetra = {0: V4, 1: E4, 2: F4}
solid_tetra  = {0: V4, 1: E4, 2: F4, 3: [(1, 2, 3, 4)]}

complexes = {"hollow_triangle": hollow_triangle, "filled_triangle": filled_triangle,
             "hollow_tetra": hollow_tetra, "solid_tetra": solid_tetra}

def rank(M):
    return 0 if M.size == 0 else int(np.linalg.matrix_rank(M))


In [ ]:
def boundary_matrix(cx, k):
    """Signed boundary matrix: rows = cx[k-1], columns = cx[k] (k >= 1).
    Hint: build a dict {simplex: row index}; for column j and position i, put (-1)**i
    in the row of the face s[:i] + s[i+1:]."""
    # TODO
    raise NotImplementedError

def betti(cx):
    """Return [beta_0, ..., beta_top]. Hint: top = max(cx); do not call boundary_matrix(cx, top + 1)."""
    # TODO
    raise NotImplementedError


In [ ]:
# GIVEN: look at your matrices
print("d1 of the filled triangle:\n", boundary_matrix(filled_triangle, 1))
print("d2 of the filled triangle:", boundary_matrix(filled_triangle, 2).ravel())
print("d1 @ d2 =", (boundary_matrix(filled_triangle, 1) @ boundary_matrix(filled_triangle, 2)).ravel())
print()
for name, cx in complexes.items():
    print(f"{name:16s} betti = {betti(cx)}")


**Bug hunt.** Below is a *broken* version of `boundary_matrix`. Run it and look at the product $\partial_1\partial_2$. Which line is wrong? (Not graded — discuss with your neighbour.)

In [ ]:
def boundary_matrix_buggy(cx, k):
    rows, cols = cx[k - 1], cx[k]
    index = {s: i for i, s in enumerate(rows)}
    M = np.zeros((len(rows), len(cols)), dtype=int)
    for j, s in enumerate(cols):
        for i in range(len(s)):
            M[index[s[:i] + s[i + 1:]], j] = 1
    return M

print("buggy d1 @ d2 =", (boundary_matrix_buggy(filled_triangle, 1) @ boundary_matrix_buggy(filled_triangle, 2)).ravel())


In [ ]:
# CHECK -- this cell must run without error
expected_d1 = np.array([[-1, -1, 0], [1, 0, -1], [0, 1, 1]])
assert np.array_equal(boundary_matrix(filled_triangle, 1), expected_d1), "d1 of the filled triangle is wrong"
assert np.array_equal(boundary_matrix(filled_triangle, 2).ravel(), [1, -1, 1]), "d2 of the filled triangle is wrong"
for name, cx in complexes.items():
    for k in range(1, max(cx)):
        assert not (boundary_matrix(cx, k) @ boundary_matrix(cx, k + 1)).any(), f"d o d != 0 for {name}, k={k}"
assert betti(hollow_triangle) == [1, 1]
assert betti(filled_triangle) == [1, 0, 0]
assert betti(hollow_tetra)    == [1, 0, 1]
assert betti(solid_tetra)     == [1, 0, 0, 0]
print("Exercise 1 passed.")


In [ ]:
# GIVEN: a picture (works once your functions are correct)
fig, ax = plt.subplots(1, 3, figsize=(11, 3.2), gridspec_kw={"width_ratios": [1, 1, 1.6]})
P = np.array([[0, 0], [1, 0], [0, 1]])
for a, (title, fill) in zip(ax[:2], [("hollow triangle", False), ("filled triangle", True)]):
    if fill: a.fill(P[:, 0], P[:, 1], color=ICE)
    a.plot(*np.vstack([P, P[:1]]).T, color=TEAL, lw=3)
    a.scatter(P[:, 0], P[:, 1], color=NAVY, zorder=3, s=40)
    a.set_title(title); a.set_aspect("equal"); a.axis("off")
names = list(complexes)
width = 0.2
for k in range(4):
    vals = [(betti(complexes[n]) + [0] * 4)[k] for n in names]
    bars = ax[2].bar(np.arange(len(names)) + (k - 1.5) * width, vals, width, label=f"$\\beta_{k}$",
                     color=[NAVY, TEAL, AMBER, ICE][k])
    ax[2].bar_label(bars, fontsize=7, padding=1)
ax[2].set_xticks(range(len(names))); ax[2].set_xticklabels([n.replace("_", "\n") for n in names], fontsize=9)
ax[2].legend(frameon=False); ax[2].set_title("Betti numbers")
plt.tight_layout(); plt.show()


## Exercise 2 — Molecules

A molecule is a graph (atoms = vertices, bonds = edges; hydrogens are ignored). Its **clique complex** has a simplex for every set of mutually bonded atoms: a bond is an edge, three mutually bonded atoms form a filled triangle, and so on.

1. **Before running anything**, fill in your predicted $\beta_1$ for every molecule in `predictions`.
2. Build the clique complex of each molecule with the given `clique_complex` and compute $\beta_1$ with your `betti`. Store the results in the dictionary `beta1`.


In [ ]:
# GIVEN
def clique_complex(G, max_dim=2):
    cx = {0: [(v,) for v in sorted(G.nodes)]}
    for k in range(1, max_dim + 1):
        cx[k] = sorted(tuple(sorted(c)) for c in nx.enumerate_all_cliques(G) if len(c) == k + 1)
    return {k: v for k, v in cx.items() if v}

molecules = {
    "hexane":       [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5)],
    "cyclopropane": [(0, 1), (1, 2), (0, 2)],
    "cyclobutane":  [(0, 1), (1, 2), (2, 3), (3, 0)],
    "benzene":      [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 0)],
    "naphthalene":  [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 0), (4, 6), (6, 7), (7, 8), (8, 9), (9, 5)],
}


In [ ]:
# YOUR PREDICTIONS: the number of independent rings, as a chemist would count them
predictions = {"hexane": None, "cyclopropane": None, "cyclobutane": None, "benzene": None, "naphthalene": None}


In [ ]:
beta1 = {}
# TODO: for every molecule build clique_complex(nx.Graph(bonds)), compute betti(...) and store beta_1 in beta1[name]
print(beta1)


In [ ]:
# GIVEN: compare with your predictions (not graded)
for name in molecules:
    ok = "ok" if predictions[name] == beta1[name] else "DIFFERENT"
    print(f"{name:13s} predicted {predictions[name]}   computed {beta1[name]}   {ok}")


In [ ]:
# CHECK
assert beta1 == {"hexane": 0, "cyclopropane": 0, "cyclobutane": 1, "benzene": 1, "naphthalene": 2}
print("Exercise 2 passed.")


In [ ]:
# GIVEN: pictures
fig, axes = plt.subplots(1, 5, figsize=(13, 2.6))
for a, (name, bonds) in zip(axes, molecules.items()):
    G = nx.Graph(bonds)
    pos = nx.kamada_kawai_layout(G)
    nx.draw(G, pos, ax=a, node_color=NAVY, edge_color=TEAL, width=2.5, node_size=160)
    for tri in clique_complex(G).get(2, []):
        a.fill(*zip(*[pos[v] for v in tri]), color=ICE, zorder=0)
    a.set_title(f"{name}\n$\\beta_1$ = {beta1[name]}", fontsize=10)
plt.tight_layout(); plt.show()


*Discuss:* cyclopropane is a ring, yet $\beta_1=0$. Why? Is the clique rule a fair model of a three-membered ring?

## Exercise 3 — Volumes, and where sampling fails

The unit ball in $\mathbb R^d$ sits inside the cube $[-1,1]^d$. From the lecture, the fraction of the cube it fills is
$$\rho_d=\frac{V_d}{2^d}=\frac{\pi^{d/2}}{\Gamma(d/2+1)\,2^d}.$$

1. Write `exact_fraction(d)` (use `math.gamma`).
2. Write `mc_fraction(d, n=200_000, seed=1)`: draw `n` points uniformly in $[-1,1]^d$, count how many satisfy $x_1^2+\dots+x_d^2\le1$, and return `(hits, hits / n)`.


In [ ]:
def exact_fraction(d):
    # TODO
    raise NotImplementedError

def mc_fraction(d, n=200_000, seed=1):
    # TODO: return (hits, hits / n)
    raise NotImplementedError


In [ ]:
# GIVEN
print("  d |   hits | MC estimate |   exact")
for d in [2, 5, 10, 15, 20]:
    hits, est = mc_fraction(d)
    print(f"{d:3d} | {hits:6d} | {est:.3e}   | {exact_fraction(d):.3e}")


In [ ]:
# CHECK
assert abs(exact_fraction(2) - math.pi / 4) < 1e-12
assert abs(exact_fraction(3) - math.pi / 6) < 1e-12
assert abs(exact_fraction(10) / 2.4904e-3 - 1) < 1e-3
assert abs(mc_fraction(5)[1] / exact_fraction(5) - 1) < 0.05, "Monte Carlo at d = 5 should be within 5%"
assert mc_fraction(20)[0] <= 1, "at d = 20 almost no sample falls inside the ball"
print("Exercise 3 passed.")


In [ ]:
# GIVEN: the picture
ds3 = list(range(2, 21))
res = [mc_fraction(d) for d in ds3]
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.semilogy(ds3, [exact_fraction(d) for d in ds3], color=TEAL, lw=2.5, label="exact formula")
ax.semilogy([d for d, (h, e) in zip(ds3, res) if h > 0], [e for d, (h, e) in zip(ds3, res) if h > 0],
            "o", color=AMBER, label="Monte Carlo, 200 000 points")
zero = [d for d, (h, e) in zip(ds3, res) if h == 0]
ax.plot(zero, [1e-9] * len(zero), "x", color="crimson", ms=8, label="0 hits (no estimate)")
ax.set_ylim(5e-10, 2); ax.set_xlabel("dimension d"); ax.set_ylabel("fraction of the cube")
ax.legend(frameon=False); plt.tight_layout(); plt.show()


*Discuss:* at $d=20$ the expected number of hits is $200\,000\cdot\rho_{20}\approx0.005$. What does this say about estimating small volumes by sampling?

## Exercise 4 — Distances in high dimensions

**(a) Mean and spread.** Write `distance_stats(d, n=1000, seed=0)`: draw `n` points and one query point from the standard Gaussian in $\mathbb R^d$, compute the `n` distances from the query to the points, and return `(mean, std)` of these distances. Then, for the dimensions in `ds`, collect the results in `stats_` and compute `slope_a`, the slope of $\log(\text{mean})$ against $\log d$ (use `np.polyfit(..., 1)[0]`).

**(b) Nearest-neighbour contrast.** The *contrast* of a query is $(D_{\max}-D_{\min})/D_{\min}$: the farthest distance relative to the nearest. Write `contrast(d, n=1000, runs=10, seed=0)`: in each of `runs` repetitions draw `n` points and a query uniformly from $[0,1]^d$ and compute the contrast; return the average. Collect the values for `dg` in `cs`, and compute `slope_b`, the log–log slope of contrast against $d$.


In [ ]:
def distance_stats(d, n=1000, seed=0):
    # TODO: return (mean, std) of the distances from one Gaussian query to n Gaussian points
    raise NotImplementedError

def contrast(d, n=1000, runs=10, seed=0):
    # TODO: average of (Dmax - Dmin) / Dmin over `runs` uniform data sets
    raise NotImplementedError


In [ ]:
ds = [10, 20, 50, 100, 200, 500, 1000]
stats_ = None    # TODO: list of (mean, std), one per d in ds
slope_a = None   # TODO

dg = [30, 100, 300, 1000]
cs = None        # TODO: list of contrasts, one per d in dg
slope_b = None   # TODO


In [ ]:
# GIVEN
print("    d | mean D | std D | sqrt(2d)")
for d, (m, s) in zip(ds, stats_):
    print(f"{d:5d} | {m:6.2f} | {s:5.3f} | {math.sqrt(2 * d):6.2f}")
print(f"slope of log(mean D) against log d: {slope_a:.3f}")
print()
print("contrast:", [(d, round(c, 3)) for d, c in zip(dg, cs)], f"  slope: {slope_b:.3f}")


In [ ]:
# CHECK
assert 0.45 < slope_a < 0.55, f"mean distance should grow like d^0.5, got exponent {slope_a:.3f}"
assert all(0.7 < s < 1.0 for d, (m, s) in zip(ds, stats_) if d >= 20), "the spread should stay O(1)"
assert -0.75 < slope_b < -0.45, f"contrast should decay like d^-0.5, got exponent {slope_b:.3f}"
print("Exercise 4 passed.")


In [ ]:
# GIVEN: the pictures
fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
ax[0].loglog(ds, [m for m, s in stats_], "o-", color=AMBER, label="mean D")
ax[0].loglog(ds, [math.sqrt(2 * d) for d in ds], "--", color=TEAL, label=r"$\sqrt{2d}$")
ax[0].set_xlabel("d"); ax[0].legend(frameon=False); ax[0].set_title("the mean grows")
ax[1].semilogx(ds, [s for m, s in stats_], "o-", color=AMBER)
ax[1].axhline(math.sqrt(0.75), ls="--", color=TEAL); ax[1].set_ylim(0, 1.5)
ax[1].set_xlabel("d"); ax[1].set_title("the spread does not (dashed: 0.87)")
ax[2].loglog(dg, cs, "o-", color=AMBER, label="measured")
ax[2].loglog(dg, [cs[0] * (dg[0] / d) ** 0.5 for d in dg], "--", color=TEAL, label="slope -1/2")
ax[2].set_xlabel("d"); ax[2].legend(frameon=False); ax[2].set_title("contrast (D_max - D_min) / D_min")
plt.tight_layout(); plt.show()

rng = np.random.default_rng(1)
plt.figure(figsize=(6.5, 3.4))
for d, col in zip([2, 10, 100, 1000], ["#9DB7E8", TEAL, AMBER, NAVY]):
    D = np.linalg.norm(rng.normal(size=(3000, d)) - rng.normal(size=d), axis=1)
    plt.hist(D / D.mean(), bins=60, histtype="step", lw=2, color=col, density=True, label=f"d = {d}")
plt.xlabel("D / mean D"); plt.legend(frameon=False); plt.title("distances relative to their mean")
plt.tight_layout(); plt.show()


## Exercise 5 — Real data against its shuffled copy (take-home)

`load_digits()` gives 1797 handwritten digits, each an $8\times8$ image, i.e. a point in $\mathbb R^{64}$. If we shuffle every column (pixel) *independently*, each pixel keeps its own histogram but all relationships between pixels are destroyed. Does the real data differ from its shuffled copy?

1. `variance_in_top(X, k=10)`: the fraction of the total variance captured by the first `k` PCA components (`PCA().fit(X).explained_variance_ratio_`).
2. `shuffle_columns(X, seed=0)`: return a copy of `X` in which every column is permuted independently (`rng.permutation`).
3. `contrast_samples(X, runs=300, seed=0)`: pick `runs` random rows `i` (use `rng.integers(len(X), size=runs)`); for each, compute the distances from `X[i]` to all *other* rows (remove the distance to itself) and record $(D_{\max}-D_{\min})/D_{\min}$. Return the array of contrasts. (If two images are identical the contrast is `inf`; that is fine, we use the median.)


In [ ]:
# GIVEN
X = load_digits().data.astype(float)
labels = load_digits().target
print(X.shape, "pixel values from", X.min(), "to", X.max())


In [ ]:
def variance_in_top(X, k=10):
    # TODO
    raise NotImplementedError

def shuffle_columns(X, seed=0):
    # TODO
    raise NotImplementedError

def contrast_samples(X, runs=300, seed=0):
    # TODO: return an array of `runs` contrasts
    raise NotImplementedError


In [ ]:
# GIVEN: uses your functions
Xs = shuffle_columns(X)
var_real, var_shuf = variance_in_top(X), variance_in_top(Xs)
c_real, c_shuf = contrast_samples(X), contrast_samples(Xs)
print(f"variance in the top 10 components : real {var_real:.3f}   shuffled {var_shuf:.3f}")
print(f"median nearest-neighbour contrast  : real {np.median(c_real):.2f}    shuffled {np.median(c_shuf):.2f}")


In [ ]:
# CHECK
assert var_real > 0.65 and var_shuf < 0.45, (var_real, var_shuf)
assert np.median(c_real) > 2 and np.median(c_shuf) < 1.3, (np.median(c_real), np.median(c_shuf))
print("Exercise 5 passed.")


In [ ]:
# GIVEN: pictures
fig, ax = plt.subplots(1, 4, figsize=(15, 3.4))
cum = lambda Z: np.cumsum(PCA().fit(Z).explained_variance_ratio_)[:30]
ax[0].plot(range(1, 31), cum(X), "o-", color=AMBER, ms=3, label="real")
ax[0].plot(range(1, 31), cum(Xs), "o-", color=TEAL, ms=3, label="shuffled")
ax[0].set_xlabel("number of components"); ax[0].set_ylabel("cumulative variance"); ax[0].legend(frameon=False)
for a, Z, title in [(ax[1], X, "real digits: first two PCs"), (ax[2], Xs, "shuffled: first two PCs")]:
    Y = PCA(2).fit_transform(Z)
    if Z is X: a.scatter(Y[:, 0], Y[:, 1], c=labels, cmap="tab10", s=5)
    else:      a.scatter(Y[:, 0], Y[:, 1], color="gray", s=5)
    a.set_title(title); a.set_xticks([]); a.set_yticks([])
fin = lambda c: c[np.isfinite(c)]
ax[3].hist(fin(c_real), bins=40, alpha=.7, color=AMBER, label="real", range=(0, 8))
ax[3].hist(fin(c_shuf), bins=40, alpha=.7, color=TEAL, label="shuffled", range=(0, 8))
ax[3].set_xlabel("contrast"); ax[3].legend(frameon=False)
plt.tight_layout(); plt.show()


*Discuss:* the shuffled copy has exactly the same one-pixel statistics as the real data. What, then, is the difference between the two? Which of the two looks like a point cloud near a low-dimensional manifold?